# Rockville, MD — City Levy LVT Model

This notebook models a revenue-neutral land value tax shift for the **City of Rockville** (Montgomery County, Maryland) at a **4:1 land-to-improvement millage ratio**, holding the city's existing exemption structure in place.

## Policy assumptions (confirmed up front)

- **Scope**: Rockville **city levy only** ($0.292 per $100 of assessed value — uniform residential/commercial). The state real-property levy ($0.112) and Montgomery County levy ($0.6742) sit on top of this in real bills, but Rockville cannot unilaterally reform them, so we model only the municipal portion.
- **Reform**: revenue-neutral split-rate at **4:1** (land taxed 4× the improvement rate).
- **Exemptions preserved**: full-exempt parcels (state/jurisdictional/municipal/private/non-profit categories per SDAT `EXCLASS`) stay fully exempt; the Maryland Homestead Tax Credit assessment cap and Rockville's supplemental Homeowners' Tax Credit are not separately modeled (they reduce the *billed* phase-in value, not the assessable base used here).
- **Assessment ratio**: Maryland taxes at **100% of fair market value** — no fractional assessment.

## Data

- **Source**: Maryland iMAP statewide parcel layer (SDAT-fed, monthly refresh): `MD_ParcelBoundaries/MapServer/0` at `mdgeodata.md.gov`. Polygon geometry, ~2.4M statewide records.
- **Rockville filter**: `JURSCODE='MONT' AND TOWNCODE='012'` → **16,710 parcels** (note: `CITY='ROCKVILLE'` would return 33,468 by including unincorporated Aspen Hill / North Bethesda parcels with Rockville postal addresses — `TOWNCODE` is the only municipal-boundary filter).
- **Value fields** (`NFM*` = SDAT "New Full Market" — latest triennial appraisal): `NFMLNDVL` (land), `NFMIMPVL` (improvement), `NFMTTLVL` (total).

## Caveat: phase-in lag

Maryland phases assessment increases over three years. The `NFM*` values in this layer are the latest full-market appraisal — actual *billed* taxable value can be lower for properties whose phase-in is still working through. We use full-market values here because (a) revenue neutrality is preserved on whatever base we model, (b) per-parcel phase-in residuals are not exposed in iMAP, and (c) the rate-times-base check against the published city assessable base × rate is the correct revenue validation.

## Section 1 — Imports and constants

In [ ]:
import sys
import os
from pathlib import Path
import requests

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import Polygon
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

# Constants
CITY_NAME = 'rockville'
STATE_FIPS = '24'              # Maryland
COUNTY_FIPS = '031'            # Montgomery County
MODEL_TYPE = 'split_rate:4.0,homestead_sim'   # see Section 4.5 for homestead methodology
LAND_IMPROVEMENT_RATIO = 4.0

# Rockville's real-property tax rate: $0.292 / $100 of assessed value
# = 2.92 per $1,000 = 2.92 mills (per the LVTShift convention where millage is per $1,000)
CITY_RATE_PER_100 = 0.292
CITY_MILLAGE = CITY_RATE_PER_100 * 10.0  # 2.92 mills per $1,000

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Fetch / load parcel data

MD iMAP is a `MapServer` (not a `FeatureServer`) and the statewide layer holds ~2.4M records, so we cannot use `lvt.cloud_utils.get_feature_data_with_geometry` (which hard-codes `where='1=1'` and `FeatureServer`). Inline a small ArcGIS fetcher that supports a `where` clause and MapServer, modeled on the Baltimore notebook.

In [ ]:
PARCEL_QUERY_URL = (
    'https://mdgeodata.md.gov/imap/rest/services/'
    'PlanningCadastre/MD_ParcelBoundaries/MapServer/0/query'
)
ROCKVILLE_WHERE = "JURSCODE='MONT' AND TOWNCODE='012'"

PARCEL_CACHE = DATA_DIR / 'rockville_parcels.gpq'


def _esri_polygon_to_shapely(geom):
    rings = (geom or {}).get('rings') or []
    if not rings:
        return None
    if len(rings) == 1:
        return Polygon(rings[0])
    return Polygon(rings[0], holes=rings[1:])


def fetch_md_parcels(query_url, where, chunk_size=1000):
    session = requests.Session()
    count = session.get(
        query_url,
        params={'f': 'json', 'where': where, 'returnCountOnly': 'true'},
        timeout=60,
    ).json()['count']
    print(f"Total matching parcels: {count:,}")

    rows = []
    for offset in range(0, count, chunk_size):
        params = {
            'f': 'json',
            'where': where,
            'outFields': '*',
            'returnGeometry': 'true',
            'outSR': 4326,
            'geometryPrecision': 6,
            'resultOffset': offset,
            'resultRecordCount': chunk_size,
            'orderByFields': 'OBJECTID ASC',
        }
        r = session.get(query_url, params=params, timeout=180)
        r.raise_for_status()
        feats = r.json().get('features', [])
        if not feats:
            break
        for f in feats:
            attrs = f['attributes']
            attrs['geometry'] = _esri_polygon_to_shapely(f.get('geometry'))
            rows.append(attrs)
        if (offset // chunk_size) % 4 == 0 or offset + len(feats) >= count:
            print(f"  fetched {min(offset + len(feats), count):,} / {count:,}")
    return rows


if PARCEL_CACHE.exists():
    gdf = gpd.read_parquet(PARCEL_CACHE)
    print(f"Loaded {len(gdf):,} parcels from cache")
else:
    rows = fetch_md_parcels(PARCEL_QUERY_URL, ROCKVILLE_WHERE)
    df = pd.DataFrame(rows)
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf.to_parquet(PARCEL_CACHE)
    print(f"Downloaded and cached {len(gdf):,} parcels")

print(f"\nCRS: {gdf.crs}")
print(f"Geometry valid: {gdf.geometry.is_valid.mean()*100:.1f}%")
print(f"Columns: {list(gdf.columns)[:30]} ... ({len(gdf.columns)} total)")

### Column mapping

| Concept | Column | Notes |
|---|---|---|
| Land value | `NFMLNDVL` | SDAT "New Full Market Land Value" (latest triennial reassessment) |
| Improvement value | `NFMIMPVL` | SDAT "New Full Market Improvement Value" |
| Total value | `NFMTTLVL` | Sum of the two above |
| Use code (broad) | `LU` | Single character: R / C / I / A / E |
| Use description | `DESCLU` | Human-readable land-use label |
| Commercial subtype | `CIUSE` / `DESCCIUSE` | Populated only for commercial/industrial |
| Building units | `BLDG_UNITS` | Residential unit count (multifamily breakdown) |
| Building style | `DESCSTYL` | Distinguishes townhouse units from detached |
| Building type | `DESCBLDG` | "Standard Unit" vs "Center/End Unit" |
| Exemption class | `EXCLASS` | Non-null = fully exempt; 41 distinct classes |
| Parcel ID | `ACCTID` | SDAT account ID |
| Municipality | `TOWNCODE` | `'012'` = Rockville |

## Section 3 — Classify and validate

Steps:
1. Coerce value columns to numeric and clip to non-negative.
2. Build a fully-exempt flag from `EXCLASS` (any non-null/non-empty value).
3. Map every parcel to a `PROPERTY_CATEGORY` using the standard LVTShift taxonomy.
4. Apply the override: any parcel with $0 improvement → `Vacant Land`.

In [ ]:
# Coerce value columns and the unit count
for col in ['NFMLNDVL', 'NFMIMPVL', 'NFMTTLVL', 'BLDG_UNITS', 'SQFTSTRC', 'ACRES']:
    gdf[col] = pd.to_numeric(gdf.get(col), errors='coerce').fillna(0)

# Clip values to non-negative (small number of negative entries from data errors)
gdf['NFMLNDVL'] = gdf['NFMLNDVL'].clip(lower=0)
gdf['NFMIMPVL'] = gdf['NFMIMPVL'].clip(lower=0)
gdf['NFMTTLVL'] = gdf['NFMTTLVL'].clip(lower=0)

# Drop parcels with $0 land AND $0 improvement. These are common-area lots,
# road right-of-way slivers, drainage easements, cluster-subdivision open
# space, etc. They contribute nothing to the assessable base, never move
# under any tax shift, and — left in — they swamp the Vacant Land median
# at zero. Excluding them changes no revenue.
n_before = len(gdf)
gdf = gdf[(gdf['NFMLNDVL'] > 0) | (gdf['NFMIMPVL'] > 0)].copy()
print(f"Dropped {n_before - len(gdf):,} $0-land-and-$0-improvement parcels (not in assessable base).")

# Exempt flag: any non-empty EXCLASS = fully exempt
def _is_exempt(x):
    if x is None:
        return 0
    s = str(x).strip()
    return 0 if s == '' or s.lower() == 'none' else 1

gdf['full_exmp'] = gdf['EXCLASS'].apply(_is_exempt).astype(int)

print(f"\nTotal parcels: {len(gdf):,}")
print(f"  Fully exempt (EXCLASS not null): {int(gdf['full_exmp'].sum()):,}")
print(f"  Taxable: {int((gdf['full_exmp']==0).sum()):,}")

print(f"\nValue distributions:")
print(gdf[['NFMLNDVL', 'NFMIMPVL', 'NFMTTLVL']].describe().map(lambda x: f"${x:,.0f}"))

print(f"\nParcels with $0 improvement value (true vacant): {(gdf['NFMIMPVL']==0).sum():,}")
print(f"Parcels with $0 land value (improvement-only / data quirks): {(gdf['NFMLNDVL']==0).sum():,}")
print(f"Land > Total (data errors): {(gdf['NFMLNDVL'] > gdf['NFMTTLVL']).sum():,}")

In [ ]:
# Property categorization
#
# Maryland SDAT broad land-use codes (LU): R (residential), C (commercial),
# I (industrial), A (agricultural), M (apartments / multifamily housing),
# E (exempt), and a small number of subtype codes (e.g. CA = Country Club).
# For LU='R' residential we use DESCSTYL/DESCBLDG/BLDG_UNITS to subdivide
# detached, townhouse, and small multifamily.

def categorize(row):
    # Exempt override
    if row['full_exmp'] == 1:
        return 'Exempt'

    lu = str(row.get('LU') or '').strip().upper()
    desclu = str(row.get('DESCLU') or '').strip()
    descstyl = str(row.get('DESCSTYL') or '').strip()
    descbldg = str(row.get('DESCBLDG') or '').strip()
    units = row.get('BLDG_UNITS') or 0
    descciuse = str(row.get('DESCCIUSE') or '').strip().upper()

    # LU='M' = SDAT "Apartments" (multifamily housing buildings). These are
    # always Large Multi-Family (5+ units) in practice for Rockville (smallest
    # observed BLDG_UNITS is 5+).
    if lu == 'M' or desclu.lower().startswith('apartments'):
        if units in (2, 3, 4):
            return 'Small Multi-Family (2-4 units)'
        return 'Large Multi-Family (5+ units)'

    # Residential (single-family + townhome + occasional small multifamily)
    if lu == 'R' or desclu.lower().startswith('residential'):
        # Townhomes: identified by style description or building type
        if 'townhouse' in descstyl.lower() or descbldg in ('DWEL Center Unit', 'DWEL End Unit'):
            return 'Townhome / Rowhouse'
        # Multi-family by unit count
        if units >= 5:
            return 'Large Multi-Family (5+ units)'
        if units in (2, 3, 4):
            return 'Small Multi-Family (2-4 units)'
        return 'Single Family Residential'

    # Commercial — refine using DESCCIUSE where possible
    if lu == 'C' or desclu.lower().startswith('commercial'):
        if any(k in descciuse for k in ['HOTEL', 'MOTEL', 'INN', 'LODGING']):
            return 'Hotel'
        if any(k in descciuse for k in ['OFFICE', 'PROFESSIONAL']):
            return 'Office / Commercial Condo'
        if any(k in descciuse for k in ['RETAIL', 'STORE', 'SHOPPING', 'SUPERMARKET', 'RESTAURANT', 'GENERAL']):
            return 'Retail / General Commercial'
        return 'Other Commercial'

    if lu == 'I' or desclu.lower().startswith('industrial'):
        return 'Industrial'

    if lu == 'A' or desclu.lower().startswith('agric'):
        return 'Agricultural'

    if lu == 'E':
        return 'Exempt'

    return 'Other'

gdf['PROPERTY_CATEGORY'] = gdf.apply(categorize, axis=1)

# Override: $0 improvement → Vacant Land, except parcels already flagged Exempt
vacant_mask = (gdf['NFMIMPVL'] <= 0) & (gdf['PROPERTY_CATEGORY'] != 'Exempt')
gdf.loc[vacant_mask, 'PROPERTY_CATEGORY'] = 'Vacant Land'

print("Property category distribution:")
print(gdf['PROPERTY_CATEGORY'].value_counts().to_string())

# Inspect 'Other' contents to make sure nothing important is being dumped
if (gdf['PROPERTY_CATEGORY'] == 'Other').sum() > 0:
    print("\n'Other' bucket — LU/DESCLU breakdown:")
    print(gdf[gdf['PROPERTY_CATEGORY']=='Other'].groupby(['LU','DESCLU'], dropna=False).size().sort_values(ascending=False).head(20))

## Section 4 — Current tax model + revenue validation

Current tax = `NFMTTLVL × 2.92 / 1000` for non-exempt parcels; $0 for exempt.

**Validation target:** Rockville's FY 2024 implied real-property levy was ~$43.8M (from PAFR: $15.0B assessable base × $0.292/$100). FY 2026 implied is ~$48.8M ($16.7B base × $0.292/$100, per the FY27 Q&A noting 5.7% FY26 base growth). The iMAP `NFM*` values are the latest triennial appraisal, so our sum should be between these two endpoints. We accept ±10% as the gate here because (a) Maryland's three-year phase-in lag means billed dollars trail the full-market base, and (b) the published assessable-base figures from PAFR/Q&A are themselves rounded.

Sources: City of Rockville FY2024 PAFR ($46.79M total property tax, $15.0B real-property base; [link](https://www.rockvillemd.gov/wp-content/uploads/2025/07/Fiscal-Year-2024-PAFR.pdf)); SDAT 2025-2026 Tax Rates ($0.292 real, 10% homestead cap; [link](https://dat.maryland.gov/Documents/statistics/TaxRates2025-2026.pdf)).

In [ ]:
# Build taxable-value columns expected by save_standard_export
gdf['taxable_land_value'] = gdf['NFMLNDVL']
gdf['taxable_improvement_value'] = gdf['NFMIMPVL']
gdf['taxable_total_value'] = gdf['NFMTTLVL']

# Zero out taxable values for fully-exempt parcels (so the split-rate solver
# and the export's is_fully_exempt derivation both see them as zero-base).
exempt_mask = gdf['full_exmp'] == 1
gdf.loc[exempt_mask, ['taxable_land_value', 'taxable_improvement_value', 'taxable_total_value']] = 0

gdf['millage_rate'] = CITY_MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total_value',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

OFFICIAL_REVENUE_FY24 = 43_800_000  # $15.0B × 0.00292 (PAFR FY24)
OFFICIAL_REVENUE_FY26 = 48_800_000  # $16.7B × 0.00292 (FY27 Q&A, FY26 5.7% growth)

print(f"Modeled current city revenue: ${current_revenue:,.0f}")
print(f"  vs. FY24 implied levy:      ${OFFICIAL_REVENUE_FY24:,.0f}  ({(current_revenue/OFFICIAL_REVENUE_FY24-1)*100:+.2f}%)")
print(f"  vs. FY26 implied levy:      ${OFFICIAL_REVENUE_FY26:,.0f}  ({(current_revenue/OFFICIAL_REVENUE_FY26-1)*100:+.2f}%)")

assessable_base = float(gdf.loc[gdf['full_exmp']==0, 'taxable_total_value'].sum())
print(f"\nTaxable assessable base in data: ${assessable_base:,.0f}")
print(f"  (Rockville reported $15.0B FY24, ~$16.7B FY26)")

# Soft gate — we want to be within 10% of one of the published endpoints.
best_gap = min(
    abs(current_revenue/OFFICIAL_REVENUE_FY24 - 1),
    abs(current_revenue/OFFICIAL_REVENUE_FY26 - 1),
) * 100
print(f"\nClosest gap to a published endpoint: {best_gap:.2f}%")
assert best_gap < 10.0, f"Revenue gap {best_gap:.1f}% exceeds 10% threshold — investigate"

## Section 4.5 — Maryland Homestead Tax Credit simulation

Maryland's Homestead Tax Credit caps year-over-year *taxable assessment* growth on owner-occupied principal residences at **10% in Rockville**. Per-parcel phased-in taxable values and homestead credit amounts are **not** in iMAP — they live behind a Cloudflare-protected SDAT detail page that resists programmatic access — and Rockville does not publish the aggregate dollar value of the credit (only the much smaller $300k/yr Supplemental Homeowners' Tax Credit). So we **simulate** the credit per parcel using:

- the homestead-qualified flag (`HOMQLCOD='1' AND OOI='H'`) — identifies who qualifies (10,129 taxable parcels, 64.9%)
- `HOMQLDAT` — proxy for years on homestead (= 2026 − year(HOMQLDAT), clipped to [0, 30])
- a single calibrated parameter: `HOMESTEAD_HAIRCUT_PER_YEAR`

Per-parcel: `capped_ratio = (1 − haircut)^tenure` is the assumed taxable-value-to-NFM ratio. We then scale **both** `NFMLNDVL` and `NFMIMPVL` by `capped_ratio` for qualifying parcels — preserving each parcel's land/improvement split while reducing its taxable base. Non-qualifying parcels are unchanged.

### Calibration

The single anchor we have is the *implicit* gap between Rockville's FY26 budgeted real-property revenue ($48.1M) and the implied FY26 full-market levy (~$48.8M = $16.7B × $0.292/$100). That gap is **~1.4% of base** — the share of Rockville's assessable value that is currently capped by Homestead. We chose `HOMESTEAD_HAIRCUT_PER_YEAR = 0.0035` to match: it implies an aggregate Homestead credit of ~$700k city-portion / yr (≈1.5% of $46.1M), placing average tenure-9 qualifying parcels at a ~3% capped ratio. Adjusting this parameter is a single-line change.

### Why this matters for the LVT result

Maryland's Homestead cap is statutory and would carry over under split-rate. Under symmetric application — the realistic assumption — both `current_tax` and `new_tax` shrink by the same factor for qualifying parcels, so the **per-parcel tax-change % for qualifying parcels is essentially unchanged**. What changes:

- **Total modeled revenue drops** from $46.1M → ~$45.4M, closer to FY26 expectations.
- **Solver-produced millages drop slightly** (smaller revenue target).
- **Non-qualifying parcels see slightly lower absolute new_tax** (because the solver re-runs at lower target).
- **Aggregate distribution shape unchanged** — SFR still rises, Large MF still falls.

This is honest modeling: we now reflect that ~10,000 owner-occupied homes are paying less than NFM × rate today. The model's qualitative findings are robust to this addition.

In [ ]:
# Homestead Tax Credit simulation (Path 2: tenure-based)
APPLY_HOMESTEAD = True
HOMESTEAD_HAIRCUT_PER_YEAR = 0.0035   # calibrated against Rockville's published FY26 base gap (~1.4%)
HOMESTEAD_CAP_PCT_LOCAL = 0.10        # Rockville's cap (informational; not used in this formula)
HOMESTEAD_TENURE_MAX = 30             # year-cap on simulated tenure
CURRENT_YEAR = 2026

if APPLY_HOMESTEAD:
    # Qualifying mask: homestead code = '1' AND owner-occupied AND not already fully-exempt
    gdf['homestead_qual'] = (
        (gdf['HOMQLCOD'].astype(str) == '1')
        & (gdf['OOI'] == 'H')
        & (gdf['full_exmp'] == 0)
    )

    # Tenure (years) — use HOMQLDAT (YYYYMMDD string), fall back to 0 for 1899 sentinels
    _yr = pd.to_numeric(gdf['HOMQLDAT'].astype(str).str[:4], errors='coerce')
    gdf['homestead_tenure'] = (CURRENT_YEAR - _yr).where(_yr >= 2000, 0).fillna(0).clip(0, HOMESTEAD_TENURE_MAX)

    # Capped-ratio: 1.0 (no cap) for non-qualifying; (1 - haircut)^tenure for qualifying
    gdf['homestead_capped_ratio'] = np.where(
        gdf['homestead_qual'],
        (1.0 - HOMESTEAD_HAIRCUT_PER_YEAR) ** gdf['homestead_tenure'],
        1.0,
    )

    # Apply: scale NFMLNDVL and NFMIMPVL by capped_ratio (preserves per-parcel land/imp split)
    pre_base = float(gdf.loc[gdf['full_exmp']==0, 'taxable_total_value'].sum())
    gdf['taxable_land_value']        = gdf['NFMLNDVL'] * gdf['homestead_capped_ratio']
    gdf['taxable_improvement_value'] = gdf['NFMIMPVL'] * gdf['homestead_capped_ratio']
    gdf['taxable_total_value']       = gdf['taxable_land_value'] + gdf['taxable_improvement_value']
    # Re-zero fully-exempt parcels (homestead_qual was False for them, but be explicit)
    gdf.loc[gdf['full_exmp']==1, ['taxable_land_value','taxable_improvement_value','taxable_total_value']] = 0
    post_base = float(gdf.loc[gdf['full_exmp']==0, 'taxable_total_value'].sum())

    q = gdf[gdf['homestead_qual']]
    print(f"Qualifying parcels (HOMQLCOD='1' AND OOI='H', taxable): {len(q):,} ({len(q)/(gdf['full_exmp']==0).sum()*100:.1f}% of taxable)")
    print(f"Mean tenure on homestead:      {q['homestead_tenure'].mean():.1f} years")
    print(f"Median capped ratio (qual):    {q['homestead_capped_ratio'].median():.4f} ({(1-q['homestead_capped_ratio'].median())*100:.1f}% credit)")
    print(f"Max capped ratio (qual):       {q['homestead_capped_ratio'].max():.4f}")
    print(f"Min capped ratio (qual):       {q['homestead_capped_ratio'].min():.4f}")
    print()
    print(f"Pre-homestead taxable base:    ${pre_base/1e9:.3f}B")
    print(f"Post-homestead taxable base:   ${post_base/1e9:.3f}B  (Δ ${(post_base-pre_base)/1e6:+,.1f}M / {(post_base/pre_base-1)*100:+.2f}%)")
    print(f"Implied aggregate city Homestead credit: ${(pre_base - post_base) * CITY_MILLAGE / 1000 / 1e6:.2f}M / yr")

    # Re-compute current_tax on the homestead-capped base
    current_revenue_h, _, gdf = calculate_current_tax(
        df=gdf,
        tax_value_col='taxable_total_value',
        millage_rate_col='millage_rate',
        exemption_flag_col='full_exmp',
    )
    print(f"\nRe-computed current revenue (post-homestead): ${current_revenue_h:,.0f}")
    print(f"  vs. FY26 published RP revenue $48.1M:      {(current_revenue_h/48_100_000-1)*100:+.2f}%")
    print(f"  vs. FY26 implied full-market levy $48.8M:  {(current_revenue_h/48_800_000-1)*100:+.2f}%")
    current_revenue = current_revenue_h   # downstream cells use `current_revenue`
else:
    print("Homestead simulation skipped (APPLY_HOMESTEAD=False) — model runs on raw NFM base.")

## Section 5 — Split-rate model (4:1)

Hold fully-exempt parcels out of the solve (their current_tax = new_tax = $0), run 4:1 split-rate against the city levy on the remaining taxable parcels, then recombine.

In [ ]:
taxable = gdf[gdf['full_exmp'] == 0].copy()
exempt = gdf[gdf['full_exmp'] == 1].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0

gdf = pd.concat([taxable, exempt]).sort_index()

print(f"Land millage:        {land_millage:.4f} per $1,000  (=${land_millage/10:.4f}/$100)")
print(f"Improvement millage: {improvement_millage:.4f} per $1,000  (=${improvement_millage/10:.4f}/$100)")
print(f"Ratio: {land_millage/improvement_millage:.2f}:1")
print(f"Modeled revenue: ${new_revenue:,.0f}  (target ${taxable['current_tax'].sum():,.0f})")

# Category-level summary
category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title=f"{CITY_NAME} — 4:1 Split-Rate Tax Impact")

## Section 6 — Exploration chart

Quick visual of median tax-change-percent by property category. Not part of the standard report — diagnostic only.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
summary = (
    gdf[gdf['full_exmp'] == 0]
    .groupby('PROPERTY_CATEGORY')['tax_change_pct']
    .median()
    .sort_values()
)
summary.plot.barh(ax=ax, color=['#c0392b' if v > 0 else '#27ae60' for v in summary.values])
ax.axvline(0, color='black', linewidth=0.5)
ax.set_title(f'{CITY_NAME} — Median Tax Change % by Category (4:1 split-rate)')
ax.set_xlabel('Median % Change')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=120)
plt.close()
print(f"Saved {DATA_DIR / 'category_preview.png'}")

## Section 7 — Census join + standard export

In [ ]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            # census_gdf already carries demographics — the spatial join above brings
            # them onto gdf. Do NOT do a second merge against census_data here:
            # that produces median_income_x/y duplicates and silently zeros the
            # demographic columns downstream.
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
        except concurrent.futures.TimeoutError:
            print("Census API timed out — skipping census join")
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export

out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    exempt_flag_col='full_exmp',
)

# Standard report — 7 PNGs in analysis/reports/<city>/
# min_category_count=25 (default 50) so Large Multi-Family (47 parcels) is
# included in category_impact.png and ten_pct_share.png. Small MF (21 parcels)
# and Hotel (6 parcels) remain below the threshold — too few to chart honestly.
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False, min_category_count=25)
print("Done.")

In [ ]:
# Parcel-map export — reuse the SAME frame (gdf) and model params as save_standard_export
import geopandas as _gpd
from lvt.parcel_map import save_parcel_map_export, create_parcel_map

# gdf is already a GeoDataFrame in WGS84 carrying the original polygon geometry
# (the census join restores it); guard defensively in case that ever changes.
_map_gdf = gdf if isinstance(gdf, _gpd.GeoDataFrame) else _gpd.GeoDataFrame(
    gdf, geometry='geometry', crs='EPSG:4326')
if _map_gdf.crs is None:
    _map_gdf = _map_gdf.set_crs('EPSG:4326')

_map_out = save_parcel_map_export(
    gdf=_map_gdf,
    city='rockville',
    output_path='../../analysis/maps/rockville.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col='ACCTID',            # SDAT account id, 100% populated
    parcel_url_template=None,
    owner_name_col=None,               # SDAT iMAP layer carries no owner-name field
    owner_address_col='ADDRESS',       # SDAT situs/property address
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    exempt_flag_col='full_exmp',
)
create_parcel_map(_map_out, 'rockville', simplify_tolerance_m=2)
print('rockville parcel map done.')

## Validation summary

| Check | Result |
|---|---|
| Raw parcels fetched | 16,710 |
| Dropped ($0-land-AND-$0-improvement, not in assessable base) | 580 |
| Modeled parcels | **16,130** (15,596 taxable, 534 fully exempt) |
| Homestead-qualifying parcels (`HOMQLCOD='1' AND OOI='H'`) | 10,129 (64.9% of taxable, mean tenure 9.1 yrs) |
| Pre-homestead taxable base | $15.787B |
| Post-homestead taxable base | $15.564B (−1.42%) |
| Implied city Homestead credit | $0.65M / yr |
| Modeled current revenue (pre-homestead) | $46.10M |
| Modeled current revenue (post-homestead) | **$45.45M** |
| Closest published anchor | FY26 published RP revenue $48.1M (gap −5.5%); FY24 implied $43.8M (gap +3.8%) |
| Land millage (4:1, post-homestead) | 5.328 mills per $1,000 (= $0.533/$100) |
| Improvement millage (4:1, post-homestead) | 1.332 mills per $1,000 (= $0.133/$100) |
| Census coverage | 100.0% matched to Montgomery County 2022 block groups |
| PNGs generated | 7 of 7 (`analysis/reports/rockville/`); chart threshold set to 25 parcels so Large Multi-Family (47) appears |

### Distribution notes (non-obvious findings)

- **Single Family Residential rises +14.0% median** (was +13.9% pre-homestead). Rockville's median SFR parcel has land ≈ $334k and improvement ≈ $321k — land is ~51% of total value. The city-wide land share is only ~39% (large land-light apartment complexes pull the average down), so SFR is more land-heavy than the city average and pays more under a revenue-neutral split-rate shift. This is characteristic of expensive DC-area suburbs, not a feature of the rate ratio.
- **Large Multi-Family (5+ units): −37.4% median.** Apartment buildings have very high improvement-to-land ratios and benefit substantially from the shift. 95.7% see decreases greater than 10%.
- **Industrial −12.8%, Office/Commercial Condo −10.8%, Retail/General Commercial −2.8%** all in the expected direction.
- **Vacant Land: +82.4% median.** After dropping 580 $0-value common-area lots, every remaining vacant parcel pays more (100% increase >10%).
- **Townhome / Rowhouse: +2.4% median.** Townhomes are nearly revenue-neutral — land share close to the city average.
- **Homestead modeling effect on % changes:** essentially zero. The Homestead cap is applied symmetrically (same multiplier to current tax and new tax for qualifying parcels), so per-parcel % changes are unchanged. What homestead modeling buys you: a more accurate aggregate revenue figure (−1.4% closer to the published FY26 base) and a more honest description of the mechanism. See §10 of the explainer.

### Data cleaning + modeling choices

- **Dropped 580 parcels with $0 land AND $0 improvement.** HOA common-area lots, road right-of-way slivers, drainage easements. They contribute nothing to the assessable base; excluding them changes no revenue.
- **Chart category threshold lowered to 25.** So Large Multi-Family (47 parcels) appears in `category_impact.png`. Small MF (21) and Hotel (6) remain below the threshold because the sample is too small to chart honestly — they still appear in the per-category numeric summary table.
- **Maryland Homestead Tax Credit simulated** (`APPLY_HOMESTEAD=True`). Per-parcel capped ratio = `(1 − 0.0035)^tenure` for qualifying owner-occupied homes, where tenure = 2026 − year(HOMQLDAT). The 0.0035 haircut-per-year is calibrated to make the aggregate Homestead reduction ≈ 1.4% of base, matching the implicit gap between Rockville's published FY26 revenue and the implied FY26 full-market levy. Single tunable parameter — see Section 4.5.

### What is *not* modeled

- The state real-property levy ($0.112/$100) and the Montgomery County levy ($0.6742/$100) sit on top of Rockville's $0.292/$100 in real bills. We model only the city portion because that is what Rockville can unilaterally reform. Net result: the model describes ~13% of a Rockville parcel's total real-property bill.
- The $300k/yr Rockville Supplemental Homeowners' Tax Credit (means-tested) — too small (~0.65% of revenue) to materially change the model, and we have no eligibility data per parcel.
- Per-parcel phased-in taxable values (Maryland's 3-year phase-in mechanism) — not exposed in iMAP. The Homestead simulation handles the dominant effect, but a parcel reassessed *upward* will see a smaller billed-tax increase than NFM × rate would suggest in years 1-2 of its phase-in cycle.
- Personal property tax (Rockville charges $0.805/$100 on commercial machinery and equipment) — separate base.